In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, max as spark_max, min as spark_min


In [15]:
spark = SparkSession.builder.appName("AnalisisExploratorio").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/06/23 14:11:35 WARN Utils: Your hostname, amidala, resolves to a loopback address: 127.0.1.1; using 192.168.1.47 instead (on interface wlp5s0)
25/06/23 14:11:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Parsed arguments:
  master                  local[*]
  remote                  null
  deployMode              null
  executorMemory          null
  executorCores           null
  totalExecutorCores      null
  propertiesFile          null
  driverMemory            null
  driverCores             null
  driverExtraClassPath    null
  driverExtraLibraryPath  null
  driverExtraJavaOptions  null
  supervise               false
  queue                   null
  numExecutors            null
  files                   null
  pyFiles                 null
  archives                null
  mainClass               null
  primaryResource         pyspark-shell
  name                    Analisi

In [ ]:
# Cargar y mostrar datos de ejemplo
# Cargar el archivo CSV y mostrar las primeras filas y el esquema
df = spark.read.csv("../data/Bank_Customer_Churn_Prediction.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()


+-----------+------------+-------+------+---+------+---------+---------------+-----------+-------------+----------------+-----+
|customer_id|credit_score|country|gender|age|tenure|  balance|products_number|credit_card|active_member|estimated_salary|churn|
+-----------+------------+-------+------+---+------+---------+---------------+-----------+-------------+----------------+-----+
|   15634602|         619| France|Female| 42|     2|      0.0|              1|          1|            1|       101348.88|    1|
|   15647311|         608|  Spain|Female| 41|     1| 83807.86|              1|          0|            1|       112542.58|    0|
|   15619304|         502| France|Female| 42|     8| 159660.8|              3|          1|            0|       113931.57|    1|
|   15701354|         699| France|Female| 39|     1|      0.0|              2|          0|            0|        93826.63|    0|
|   15737888|         850|  Spain|Female| 43|     2|125510.82|              1|          1|            1|

In [ ]:
# Análisis exploratorio básico
# Mostrar el número de filas, columnas y estadísticas descriptivas
print("Cantidad de filas:", df.count())
print("Columnas:", df.columns)
df.describe().show()


Cantidad de filas: 10000
Columnas: ['customer_id', 'credit_score', 'country', 'gender', 'age', 'tenure', 'balance', 'products_number', 'credit_card', 'active_member', 'estimated_salary', 'churn']


25/06/23 14:12:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+-----------------+-------+------+------------------+------------------+-----------------+------------------+-------------------+-------------------+-----------------+-------------------+
|summary|      customer_id|     credit_score|country|gender|               age|            tenure|          balance|   products_number|        credit_card|      active_member| estimated_salary|              churn|
+-------+-----------------+-----------------+-------+------+------------------+------------------+-----------------+------------------+-------------------+-------------------+-----------------+-------------------+
|  count|            10000|            10000|  10000| 10000|             10000|             10000|            10000|             10000|              10000|              10000|            10000|              10000|
|   mean|  1.56909405694E7|         650.5288|   NULL|  NULL|           38.9218|            5.0128|76485.88928799961|            1.5302|         

In [ ]:
# Transformaciones de datos
# Filtrar clientes con saldo mayor a 100000
df_saldo_alto = df.filter(df.balance > 100000)
df_saldo_alto.show(5)

+-----------+------------+-------+------+---+------+---------+---------------+-----------+-------------+----------------+-----+
|customer_id|credit_score|country|gender|age|tenure|  balance|products_number|credit_card|active_member|estimated_salary|churn|
+-----------+------------+-------+------+---+------+---------+---------------+-----------+-------------+----------------+-----+
|   15619304|         502| France|Female| 42|     8| 159660.8|              3|          1|            0|       113931.57|    1|
|   15737888|         850|  Spain|Female| 43|     2|125510.82|              1|          1|            1|         79084.1|    0|
|   15574012|         645|  Spain|  Male| 44|     8|113755.78|              2|          1|            0|       149756.71|    1|
|   15656148|         376|Germany|Female| 29|     4|115046.74|              4|          1|            0|       119346.88|    1|
|   15792365|         501| France|  Male| 44|     4|142051.07|              2|          0|            1|

In [ ]:
# Agregaciones y estadísticas
# Saldo promedio, máximo y mínimo por país y género
df.groupBy("country", "gender").agg(
    avg("balance").alias("saldo_promedio"),
    spark_max("balance").alias("saldo_maximo"),
    spark_min("balance").alias("saldo_minimo")
).show()

+-------+------+------------------+------------+------------+
|country|gender|    saldo_promedio|saldo_maximo|saldo_minimo|
+-------+------+------------------+------------+------------+
|Germany|Female|119145.96647108134|   206868.78|    32197.64|
| France|  Male| 63546.28487468217|   212692.97|         0.0|
| France|Female|60322.670159221685|   238387.56|         0.0|
|  Spain|  Male|63352.833746397584|   250898.09|         0.0|
|Germany|  Male|120259.66822188442|   214346.96|    27288.43|
|  Spain|Female| 59862.09253443531|   216109.88|         0.0|
+-------+------+------------------+------------+------------+

